# STEM in one notebook

This walks the whole pipeline of **The Algebra of Storytelling** end to end:

```
video shots ──> VLM presence detection ──> S_obs
                                             │
                   ΔS_obs = S_obs[1:] − S_obs[:−1]  ◄──┘
                   E      = ΔS_obs − ΔS*
                                │
                   PTM · EPA · Type I–IV failures
```

The idea in one sentence: **a narrative is a sequence of presence states, so
narrative errors are differences between transition matrices.** Everything
below is arithmetic on small integer matrices.

We use **real video**, not generated video. That is deliberate: with real
footage cut to a known plan, the right answer is known in advance, so what
you see here tests the metric rather than any generator.

No GPU is required — the stored detections ship with the repo, and one switch
re-runs the detector yourself.

In [ ]:
import os
import sys
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from stem import (
    load_benchmark, evaluate_presence, transition_matrix, error_matrix,
    compute_metrics, classify_failures, classify_cells, shot_structure,
    TYPE_NAMES,
)
from stem.frame_extraction import extract_middle_frame
from span.pattern_library import PATTERNS

ASSETS = os.path.join(REPO, "demo", "assets")
VIDEOS = os.path.join(ASSETS, "gt_videos")
print(f"repo: {REPO}")


## 1. Five real videos, one plan

Three clips of two cats — one alone, both together, the other alone — recut
into five three-shot videos. All five are *supposed* to tell the same Relay
story. One does. The other four each break it in exactly one way.

In [ ]:
cases = load_benchmark(os.path.join(ASSETS, "gt_cases.json"))
meta = json.load(open(os.path.join(ASSETS, "gt_cases.json")))

print(f"{meta['prescribed_story']}\n")
print("Source footage:")
for clip, what in meta["source_clips"].items():
    print(f"  {clip:<16} {what}")

print(f"\n{'case':<26}{'shots cut from':<52}what should go wrong")
print("-" * 110)
for sid, sc in cases.items():
    print(f"{sid:<26}{' + '.join(sc['shot_sources']):<52}"
          f"{'nothing' if sid == 'GT_Perfect' else ', '.join(sc['expected']['failure_types'])}")


## 2. Take one case

`Type_I_ContextBleeding` reuses the two-cat clip for its final shot, so the
black and white cat never leaves. That is a narrative failure that no
identity-consistency metric would flag — the cat is rendered perfectly, it is
simply still there.

In [ ]:
CASE = "Type_I_ContextBleeding"
scenario = cases[CASE]
video = os.path.join(VIDEOS, f"{CASE}.mp4")

print(f"{CASE}")
print(f"  pattern  : {scenario['pattern']}  {scenario['notation']}")
print(f"  prescribed: {' -> '.join(scenario['shot_structure'])}")
print(f"  built from: {' + '.join(scenario['shot_sources'])}")
print(f"  note     : {scenario['note']}\n")
print("Entities")
for e in scenario["entities"]:
    print(f"  {e['entity_id']}: {e['name']}   (detector asks: "
          f"\"Is there a {e['eval_name']} visible in this image?\")")


## 3. The frames the detector sees

One frame per shot, taken from the middle. STEM measures *established,
shot-level* presence rather than within-shot tracking, which is why one frame
per shot is the intended granularity.

In [ ]:
frames = [extract_middle_frame(video, i) for i in range(scenario["num_shots"])]

fig, axes = plt.subplots(1, 3, figsize=(11, 6))
for i, (ax, img) in enumerate(zip(axes, frames)):
    ax.imshow(img)
    ax.set_title(f"Shot {i+1}\nprescribed: {scenario['shot_structure'][i]}", fontsize=11)
    ax.axis("off")
fig.suptitle(f"{CASE} — the frame the detector actually sees, one per shot", fontsize=12)
fig.tight_layout()
plt.show()


## 4. Detect entity presence

One binary question per (shot, entity) pair, greedy decoding, no threshold:

> *Is there a {entity} visible in this image? Answer only Yes or No.*

Stacking the answers gives the observed presence matrix `S_obs`.

In [ ]:
USE_GPU = False  # set True to run Qwen2-VL-7B yourself (~16 GB VRAM)

entity_queries = [e["eval_name"] for e in scenario["entities"]]

if USE_GPU:
    from stem import STEMEvaluator
    ev = STEMEvaluator(benchmark=os.path.join(ASSETS, "gt_cases.json"), gpu=0)
    S_obs, answers = ev.detect(video, scenario)
    S_obs = np.array(S_obs)
else:
    cached = json.load(open(os.path.join(ASSETS, "gt_detected.json")))
    S_obs = np.array(cached["cases"][CASE]["S_obs"])
    answers = cached["cases"][CASE]["answers"]
    print(f"Using the stored detections ({cached['detector']}).")
    print("Set USE_GPU = True to re-detect from the video yourself.\n")

print("One binary query per (shot, entity):")
for t in range(S_obs.shape[0]):
    for j, name in enumerate(entity_queries):
        raw = answers.get(f"s{t}_e{j}", "")
        print(f"  shot {t+1}, '{name}': {raw}")


## 5. Prescribed vs observed presence

Rows are shots in order, columns are entities in `entity_id` order (A, B).

In [ ]:
S_star = np.array(scenario["presence_matrix"])
ids = [e["entity_id"] for e in scenario["entities"]]


def show(M, title, signed=True):
    print(f"{title}")
    for row in M:
        cells = [(f"{v:+d}" if v else " 0") if signed else f" {v}" for v in row]
        print("   [" + "  ".join(cells) + " ]")


print(f"Prescribed vs observed presence   (columns: {', '.join(ids)})\n")
show(S_star, "S*      what the story prescribes", signed=False)
print(f"        -> {' -> '.join(shot_structure(S_star, ids))}\n")
show(S_obs, "S_obs   what the detector saw", signed=False)
print(f"        -> {' -> '.join(shot_structure(S_obs, ids))}")
print("\nThe black and white cat is still there in shot 3 — because shot 3 reuses")
print("the two-cat clip. This is a real edit, not a detector mistake.")


## 6. Difference the shots: transition matrices

Presence is a *state*; narrative is what happens *between* states. Taking the
finite difference over shots turns one into the other:

$$\Delta S = S[1{:}] - S[{:}{-}1], \qquad E = \Delta S_{obs} - \Delta S^{*}$$

`+1` is a prescribed entrance, `-1` an exit, `0` no change. The error matrix
`E` localises every deviation to a specific (transition, entity) cell — which
is what makes the failure nameable in step 8.

In [ ]:
dS_star = transition_matrix(S_star)
dS_obs = transition_matrix(S_obs)
E = error_matrix(dS_obs, dS_star)

print("Differencing over shots turns presence into narrative events.")
print("+1 = entrance, -1 = exit, 0 = no change.\n")
show(dS_star, "ΔS*     prescribed transitions")
show(dS_obs, "ΔS_obs  observed transitions")
print()
show(E, "E = ΔS_obs - ΔS*   the error matrix")

n_prescribed = int(np.count_nonzero(dS_star))
print(f"\n{n_prescribed} cells are prescribed (ΔS* != 0); PTM scores only those.")


## 7. PTM and EPA

**PTM** (Prescribed Transition Match) scores only the cells where `ΔS* != 0`.
That sparsity is the whole point: a story prescribing two transitions should
not be graded on cells where nothing was supposed to happen.

**EPA** (Entity Presence Accuracy) is the dense counterpart over all presence
cells. It answers "was the right cast on screen?", not "did the story happen?"

Note the gap here: EPA is 0.83 — five of six presence cells are correct — while
PTM is 0.50, because one of only two prescribed transitions never occurred.

In [ ]:
m = compute_metrics(S_obs, S_star, dS_star)

print(f"PTM      = {m['match_prescribed']:.4f}   "
      f"({int(round(m['match_prescribed'] * n_prescribed))}/{n_prescribed} "
      f"prescribed transitions executed exactly)")
print(f"EPA      = {m['epa']:.4f}   "
      f"({int(round(m['epa'] * S_star.size))}/{S_star.size} presence cells correct)")
print(f"||E||_F  = {m['frob_dist']:.4f}")


## 8. Name the failure

Every non-zero cell of `E` is attributed to exactly one of four types,
determined by what the cell was *supposed* to do:

| type | name | condition |
|---|---|---|
| I | Context Bleeding | `ΔS* = -1`, the entity refuses to leave |
| II | Amnesia | `ΔS* = +1`, the entity never arrives |
| III | Spurious Transition | `ΔS* = 0`, it changes state anyway |
| IV | Reversal | `\|E\| = 2`, the transition ran backwards |

This is what turns a score into a diagnosis.

In [ ]:
types = sorted(classify_failures(dS_star, E))
cells = classify_cells(dS_star, dS_obs, E, ids)


def signed(v):
    return f"{v:+d}" if v else "0"


print(f"Failure types: {types}  ({', '.join(TYPE_NAMES[t] for t in types)})\n")
if cells:
    for c in cells:
        print(f"- Type {c['type']}: {c['type_name']}")
        print(f"    transition {c['transition_index']} (shot {c['transition_index']} "
              f"-> {c['transition_index'] + 1}), entity {c['entity_id']}")
        print(f"    prescribed {signed(c['prescribed'])}, "
              f"observed {signed(c['observed'])}, error {signed(c['error'])}")
else:
    print("No failures: every prescribed transition was executed.")

print(f"\nWhat the case file says should happen: "
      f"{scenario['expected']['failure_types']}")


## 9. The whole thing as one picture

In [ ]:
def plot_matrix(ax, M, title, kind, entity_ids):
    """Render one matrix in the paper's Figure 1 style."""
    M = np.asarray(M)
    if kind == "presence":
        cmap, vmin, vmax = ListedColormap(["#f2f2f2", "#3c6e9f"]), 0, 1
    elif kind == "transition":
        cmap, vmin, vmax = ListedColormap(["#c65b4e", "#f2f2f2", "#3c8f5a"]), -1, 1
    else:  # error
        cmap, vmin, vmax = plt.get_cmap("RdBu_r"), -2, 2

    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax)
    for (i, j), v in np.ndenumerate(M):
        ax.text(j, i, f"{v:+d}" if (kind != "presence" and v) else f"{v}",
                ha="center", va="center", fontsize=12,
                color="white" if abs(v) > 0 and kind == "presence" else "black")
    ax.set_xticks(range(M.shape[1]), entity_ids)
    rows = ([f"shot {i+1}" for i in range(M.shape[0])] if kind == "presence"
            else [f"{i+1}→{i+2}" for i in range(M.shape[0])])
    ax.set_yticks(range(M.shape[0]), rows)
    ax.set_title(title, fontsize=11)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    return im


fig, axes = plt.subplots(1, 5, figsize=(15, 2.9))
plot_matrix(axes[0], S_star, "S*  prescribed", "presence", ids)
plot_matrix(axes[1], S_obs, "S_obs  detected", "presence", ids)
plot_matrix(axes[2], dS_star, "ΔS*", "transition", ids)
plot_matrix(axes[3], dS_obs, "ΔS_obs", "transition", ids)
plot_matrix(axes[4], E, "E = ΔS_obs − ΔS*", "error", ids)
fig.suptitle(
    f"{CASE}    PTM = {m['match_prescribed']:.2f}    EPA = {m['epa']:.2f}    "
    f"failures: {', '.join(types)}",
    fontsize=12,
)
fig.tight_layout()
plt.show()


## 10. All five cases

Each video isolates a different failure mode. Watch EPA stay in a narrow band
while PTM collapses — that is the separation the benchmark is built to expose.

In [ ]:
cached = json.load(open(os.path.join(ASSETS, "gt_detected.json")))

print(f"{'case':<26}{'PTM':>6}{'EPA':>7}   failures")
print("-" * 62)
for sid, sc in cases.items():
    rec = evaluate_presence(sc, cached["cases"][sid]["S_obs"])
    names = ", ".join(TYPE_NAMES[t] for t in rec["failure_types"])
    print(f"{sid:<26}{rec['ptm']:>6.2f}{rec['epa']:>7.2f}   {names}")

print("\nEvery one of these videos is real footage. Where STEM reports a failure,")
print("the failure is in the edit — which is what makes these cases a check on")
print("the metric itself rather than on any generator.")


Side by side, with the error matrix that produced each score:

In [ ]:
fig, axes = plt.subplots(len(cases), 4, figsize=(12, 2.7 * len(cases)))
for row, (sid, sc) in enumerate(cases.items()):
    S_i = np.array(sc["presence_matrix"])
    S_o = np.array(cached["cases"][sid]["S_obs"])
    d_i, d_o = transition_matrix(S_i), transition_matrix(S_o)
    E_i = error_matrix(d_o, d_i)
    rec = evaluate_presence(sc, cached["cases"][sid]["S_obs"])
    e_ids = [e["entity_id"] for e in sc["entities"]]

    plot_matrix(axes[row][0], S_i, "S*" if row == 0 else "", "presence", e_ids)
    plot_matrix(axes[row][1], S_o, "S_obs" if row == 0 else "", "presence", e_ids)
    plot_matrix(axes[row][2], E_i, "E" if row == 0 else "", "error", e_ids)
    axes[row][3].axis("off")
    axes[row][3].text(
        0, 0.5,
        f"{sid}\n{' → '.join(shot_structure(S_o, e_ids))}  (want "
        f"{' → '.join(sc['shot_structure'])})\n\n"
        f"PTM = {rec['ptm']:.2f}   EPA = {rec['epa']:.2f}\n"
        f"failures: {', '.join(rec['failure_types'])}",
        fontsize=9, va="center", family="monospace",
    )
fig.suptitle("Five real videos, one per failure mode", fontsize=12)
fig.tight_layout()
plt.show()


## 11. MEMS-1K

The cat videos check the metric. **MEMS-1K** is what you evaluate a system on:
1,000 three-shot scenarios over 7 narrative patterns. **Relay** and **Split**
are the algebraic basis — Theorem 3.1 shows that column permutations and
integer combinations of those two span every transition topology in
`Z^{2x3}` — and the other five are derived from them.

### Where to go next

```bash
# score presence predictions from any detector — no GPU
python scripts/evaluate_predictions.py --observed examples/predicted_presence.json

# generate videos for MEMS-1K with your own system, name them
# <scenario_id>.mp4, then:
python scripts/evaluate_directory.py --video-dir videos/my_model \
    --output-dir outputs/my_model --method my_model
python scripts/aggregate_results.py --input outputs/my_model/results.jsonl
```

In [ ]:
mems = load_benchmark()
print(f"MEMS-1K: {len(mems)} scenarios\n")

print(f"{'pattern':<18}{'class':<10}{'structure':<20}{'prescribed':>11}{'scenarios':>11}")
print("-" * 70)
for name, spec in PATTERNS.items():
    print(f"{name:<18}{spec['type']:<10}{spec['notation']:<20}"
          f"{int(np.count_nonzero(spec['delta_S_star'])):>11}{spec['scenarios']:>11}")
print(f"\n{'total':<58}{sum(p['scenarios'] for p in PATTERNS.values()):>11}")

example = mems["DM1K_777_Relay_001"]
print(f"\nExample scenario — {example['scenario_id']} ({example['pattern']})")
print(f"  entities : {[e['name'] for e in example['entities']]}")
print(f"  structure: {' -> '.join(example['shot_structure'])}")
for i in range(example["num_shots"]):
    print(f"  shot {i+1}   : {example['eval_prompt'][f'shot_{i+1}']}")
